# Imports

In [7]:
from transformers import EfficientNetForImageClassification
import pandas as pd
from datasets import load_dataset, Dataset, DatasetDict, Audio, ClassLabel, Sequence
from glob import glob 
from pydub import AudioSegment
import os
from tqdm.auto import tqdm
import requests
import json
import time
import random
from concurrent.futures import ThreadPoolExecutor, as_completed
from opensoundscape.annotations import BoxedAnnotations

# Dataset Class Exploration

In [8]:
# load model 
model = EfficientNetForImageClassification.from_pretrained(
    "DBD-research-group/EfficientNet-B1-BirdSet-XCL",
    num_channels=1,
    # cache_dir=CACHE_DIR,
    ignore_mismatched_sizes=True,
)

# get model classes
model_labels = list(model.config.id2label.values())

# load conversion from ebird codes to scientific names
ebird_df = pd.read_excel("/home/gil/comp0173/Clements_v2025-October-2025.xlsx", usecols=[1,7], index_col=1)

# get dataset classes
dataset_df = pd.read_csv("/home/gil/comp0173/BirdSet/data/DataS1/annotations_details.csv")
dataset_labels = pd.unique(dataset_df.loc[:,"scientific_name"])

/home/gil/miniconda3/envs/comp0173-opso/lib/python3.10/site-packages/huggingface_hub/file_download.py:795: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [9]:
# convert scientific name to ebird code
dataset_classes = []

for label in dataset_labels:
    try:
        dataset_classes.append(ebird_df.loc[label,"species_code"])
    except:
        print(f"{label} not found")

# identify classes present in the dataset AND in the model
TARGET_CLASSES = list(set(dataset_classes).intersection(set(model_labels)))

# Refactor dataset into HF datasets format

In [10]:
dataset_root = "/home/gil/comp0173/BirdSet/data/DataS1"

## Train

### Download files
See `extract_xcl_train_ogg_debug.py` for downloading the dataset

In [ ]:
df_all = pd.read_parquet("/home/gil/comp0173/BirdSet/data/DataS1_DT_train/ogg/metadata-full.parquet")
failed = [443298, 842661, 282934, 497924, 769107, 600216, 769104, 629813, 267870, 509691, 722620]

missing = df_all[~df_all["filepath"].map(lambda p: os.path.exists(p))]
print("Missing:", len(missing))
print(missing[["filepath"]].head())

available = df_all[df_all["filepath"].map(lambda p: os.path.exists(p))]
available.to_parquet("/home/gil/comp0173/BirdSet/data/DataS1_DT_train/metadata-train.parquet")

Missing: 11
                                                 filepath
id                                                       
722620  /home/gil/comp0173/BirdSet/data/DataS1_DT_trai...
267870  /home/gil/comp0173/BirdSet/data/DataS1_DT_trai...
509691  /home/gil/comp0173/BirdSet/data/DataS1_DT_trai...
629813  /home/gil/comp0173/BirdSet/data/DataS1_DT_trai...
769107  /home/gil/comp0173/BirdSet/data/DataS1_DT_trai...


### Load dataset

In [11]:
# test loading dataset
dataset_DT_train = load_dataset("parquet", data_files="/home/gil/comp0173/BirdSet/data/DataS1_DT_train/metadata-train.parquet")["train"]

## Test_5s

### Dataset Format Exploration

In [12]:
# see format for HSN
dataset_exploration = load_dataset("DBD-research-group/BirdSet", "HSN")#, streaming=True)
dataset_exploration

DatasetDict({
    train: Dataset({
        features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
        num_rows: 5195
    })
    test: Dataset({
        features: ['audio', 'filepath', 'start_time', 'end_time', 'low_freq', 'high_freq', 'ebird_code', 'ebird_code_multilabel', 'ebird_code_secondary', 'call_type', 'sex', 'lat', 'long', 'length', 'microphone', 'license', 'source', 'local_time', 'detected_events', 'event_cluster', 'peaks', 'quality', 'recordist', 'genus', 'species_group', 'order', 'genus_multilabel', 'species_group_multilabel', 'order_multilabel'],
        num_rows: 10296
    })
    test_5s: Dataset({
        features: [

In [17]:
dataset_exploration["test_5s"].features

{'audio': Audio(sampling_rate=32000, mono=True, decode=False, id=None),
 'filepath': Value(dtype='string', id=None),
 'start_time': Value(dtype='float64', id=None),
 'end_time': Value(dtype='float64', id=None),
 'low_freq': Value(dtype='int64', id=None),
 'high_freq': Value(dtype='int64', id=None),
 'ebird_code': ClassLabel(names=['gcrfin', 'whcspa', 'amepip', 'sposan', 'rocwre', 'brebla', 'daejun', 'foxspa', 'clanut', 'moublu', 'casfin', 'mallar3', 'herthr', 'amerob', 'yerwar', 'yelwar', 'dusfly', 'mouchi', 'orcwar', 'warvir', 'norfli'], id=None),
 'ebird_code_multilabel': Sequence(feature=ClassLabel(names=['gcrfin', 'whcspa', 'amepip', 'sposan', 'rocwre', 'brebla', 'daejun', 'foxspa', 'clanut', 'moublu', 'casfin', 'mallar3', 'herthr', 'amerob', 'yerwar', 'yelwar', 'dusfly', 'mouchi', 'orcwar', 'warvir', 'norfli'], id=None), length=-1, id=None),
 'ebird_code_secondary': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'call_type': Value(dtype='string', id=None),


In [ ]:
samples

{'audio': {'bytes': b'OggS\x00\x02\x00\x00\x00\x00\x00\x00\x00\x00\xccG\xbc&\x00\x00\x00\x00\x04X\xb2Y\x01\x1e\x01vorbis\x00\x00\x00\x00\x01\x00}\x00\x00\x00\x00\x00\x00\xb00\x01\x00\x00\x00\x00\x00\xb8\x01OggS\x00\x00\x00\x00\x00\x00\x00\x00\x00\x00\xccG\xbc&\x01\x00\x00\x00\xca\x9d\x92\xba\x0fZ\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\xff\x91\x03vorbis4\x00\x00\x00Xiph.Org libVorbis I 20200704 (Reducing Environment)\x01\x00\x00\x00\x12\x00\x00\x00ENCODER=libsndfile\x01\x05vorbis&BCV\x01\x00\x08\x00\x00\x80"L\x18\xc4\x80\xd0\x90U\x00\x00\x10\x00\x00\xa0\xac7\x96{\xc8\xbd\xf7\xde{\x81\xa8G\x14{\x88\xbd\xf7\xde{\xe3\xacG\xd0z\x88\xb9\xf7\xde{\xee\xbd\xa7\x1a{\xcb\xbd\xf7\xdes 4d\x15\x00\x00\x04\x00\x80)\x08\x9ar\xe0B\xea\xbd\xf7\x1e\x19\xe6\x11Q\x1a*\xc7\xbd\xf7\x1e\x19\x85\x890\x94\x19\x85=\x95\xdaZ\xeb!\x93\xdcB\xea=\xe7\x1e\x08\rY\x05\x00\x00\x02\x00@\x08!\x84\x14RH!\x85\x14RH!\x85\x14RH)\xa5\x98b\x8a)\xa6\x98b\xca)\xa7\x1cs\xcc1\xc7 \x83\x0e:\xe8\xa4\x93PB\t)\xa4PJ*\xa9\xa4\x

In [25]:
dataset_exploration["test_5s"]["ebird_code_multilabel"]

[[1],
 [],
 [],
 [4],
 [12],
 [],
 [],
 [2],
 [],
 [],
 [8],
 [1],
 [],
 [],
 [],
 [2],
 [],
 [1],
 [],
 [],
 [6],
 [],
 [],
 [12],
 [2, 1],
 [1],
 [],
 [2],
 [0],
 [],
 [2],
 [],
 [],
 [],
 [],
 [1],
 [2],
 [0, 4],
 [1],
 [8],
 [1],
 [],
 [],
 [],
 [],
 [12, 1],
 [],
 [],
 [],
 [],
 [1],
 [1],
 [],
 [],
 [],
 [],
 [],
 [4],
 [],
 [],
 [],
 [1],
 [],
 [],
 [],
 [1],
 [1],
 [1],
 [],
 [2],
 [],
 [2],
 [1],
 [],
 [17],
 [1],
 [],
 [4],
 [],
 [1, 2],
 [],
 [4],
 [1, 0],
 [],
 [],
 [1],
 [1],
 [],
 [],
 [],
 [],
 [],
 [2],
 [1],
 [],
 [1],
 [],
 [],
 [],
 [],
 [],
 [],
 [2],
 [],
 [],
 [1],
 [4],
 [1],
 [],
 [1],
 [],
 [1],
 [8],
 [],
 [],
 [],
 [1],
 [],
 [1],
 [],
 [2],
 [6],
 [1],
 [],
 [1, 4],
 [],
 [1],
 [1],
 [],
 [],
 [],
 [1],
 [1],
 [1],
 [],
 [],
 [0],
 [],
 [],
 [],
 [],
 [1],
 [1],
 [4],
 [0, 1],
 [14],
 [2],
 [],
 [1],
 [1],
 [],
 [1],
 [],
 [],
 [],
 [],
 [8],
 [],
 [2],
 [12],
 [],
 [1],
 [],
 [],
 [],
 [2],
 [],
 [],
 [1],
 [1],
 [1, 2],
 [1],
 [],
 [],
 [],
 [],
 [],
 [],


### Build dataset

In [6]:
test_5s_df = pd.DataFrame(columns=dataset_exploration["test_5s"].column_names)

In [7]:
# find all audio files with recordings
audio_files = glob(os.path.join(dataset_root, "audio_annots/*.flac"))

valid_f_audio = []
valid_f_annot = []

skipped_files = []

for audio_file in tqdm(audio_files):
    # find all associated annotation files
    tag_file = audio_file.replace(".flac", "-tags.csv")
    if os.path.exists(tag_file): 
        output_path = audio_file.replace("flac", "ogg")

        # convert all audio files to .ogg
        # try:
        #     audio = AudioSegment.from_file(audio_file)
        #     audio = audio.set_frame_rate(32000).set_channels(1)

        #     audio.export(output_path, format="ogg", codec="libvorbis")

        #     del audio

        # except Exception as e:
        #     print(f"Skipping {audio_file}")
        #     skipped_files.append(audio_file)

        valid_f_audio.append(output_path)
        valid_f_annot.append(tag_file)

  0%|          | 0/2280 [00:00<?, ?it/s]

In [8]:
# concatenate annotation files into .csv appropriate for splitting into 5sec clips
formatted_annots = []
for f in tqdm(valid_f_annot):
    df = pd.read_csv(f)

    df = df[df["tag"] != "UNKN"]

    if df.shape[0] == 0:
        continue

    df.rename(columns={"start": "start_time",
                       "end": "end_time",
                       "frequency_min": "low_f",
                       "frequency_max": "high_f",
                       "tag": "annotation",
                       "file_name": "audio_file"},
                       inplace=True)
        
    df["audio_file"] = df["audio_file"].apply(lambda x: os.path.join(dataset_root, "audio_annots", x))

    df_ebird_class_names = pd.merge(pd.merge(df, dataset_df[["tag", "scientific_name"]], left_on="annotation", right_on="tag"), ebird_df.reset_index(), left_on="scientific_name", right_on="scientific name")["species_code"]
    df["annotation"] = df_ebird_class_names.apply(lambda c: TARGET_CLASSES.index(c) if (not pd.isna(c) and c in TARGET_CLASSES) else pd.NA)

    df.drop(columns=["related", "overlap", "id"], inplace=True)

    formatted_annots.append(df[~df["annotation"].isna()])

df_annots = pd.concat(formatted_annots, ignore_index = True)
df_annots.to_csv(f"/home/gil/comp0173/BirdSet/data/DataS1/boxed_annotations_input_reindexed.csv")

  0%|          | 0/2280 [00:00<?, ?it/s]

In [9]:
df_annots = pd.read_csv("/home/gil/comp0173/BirdSet/data/DataS1/boxed_annotations_input_reindexed.csv", 
                        index_col=0)

annots = BoxedAnnotations(df_annots)
annots.audio_files = annots.df["audio_file"].unique()

In [10]:
df_annots

,audio_file,annotation,start_time,end_time,low_f,high_f
0,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,2.0,12.3034,12.6876,0.0000,1700.0000
1,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,12.0,11.3550,11.8250,2083.3333,3550.0000
2,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,2.0,4.3950,4.6850,150.0000,2916.6667
3,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,2.0,7.7950,7.9750,1183.3333,2883.3333
4,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,2.0,8.1650,8.3450,1250.0000,2916.6667
...,...,...,...,...,...,...
6854,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,11.9250,16.7650,1516.6667,5550.0000
6855,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,17.9450,19.5750,1950.0000,5283.3333
6856,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,0.0350,2.2250,2150.0000,4683.3333
6857,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,12.8850,13.7950,2683.3333,4416.6667


In [11]:
# split audio into five second clips 
boxed_annots, _ = annots.clip_labels(clip_duration=5, clip_overlap=0, min_label_overlap=0.25, class_subset=[i for i in range(len(TARGET_CLASSES))], return_type="classes")

In [12]:
boxed_annots_df = boxed_annots.reset_index()    \
                              .rename(columns={"labels": "ebird_code_multilabel",
                                               "file":"filepath"})

In [17]:
len(TARGET_CLASSES)

31

In [13]:
boxed_annots_df

,filepath,start_time,end_time,ebird_code_multilabel
0,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,0.0,5.0,[]
1,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,5.0,10.0,[]
2,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,10.0,15.0,[2]
3,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,15.0,20.0,[]
4,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,25.0,[]
...,...,...,...,...
6089,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,5.0,10.0,[]
6090,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,10.0,15.0,[20]
6091,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,15.0,20.0,[20]
6092,/home/gil/comp0173/BirdSet/data/DataS1/audio_a...,20.0,25.0,[]


In [14]:
for col in test_5s_df.columns:
    if col not in boxed_annots_df.columns:
        boxed_annots_df[col] = None

In [15]:
features = dataset_exploration["test_5s"].features.copy()
features["ebird_code_multilabel"]= Sequence(ClassLabel(names=TARGET_CLASSES))

test_5s_dataset = Dataset.from_pandas(boxed_annots_df)   \
                         .cast_column("audio", Audio(sampling_rate=32000, mono=True))   \
                         .cast(features)

Casting the dataset:   0%|          | 0/6094 [00:00<?, ? examples/s]

In [18]:
test_5s_dataset["ebird_code_multilabel"]

[[],
 [],
 [2],
 [],
 [],
 [],
 [],
 [],
 [12],
 [],
 [],
 [],
 [2],
 [2],
 [],
 [],
 [],
 [],
 [21],
 [],
 [],
 [20],
 [20],
 [20],
 [21],
 [21],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [],
 [],
 [],
 [20, 21],
 [20],
 [],
 [],
 [],
 [],
 [],
 [18],
 [18],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [7],
 [7],
 [7],
 [7],
 [7],
 [7],
 [20],
 [20],
 [20],
 [20],
 [],
 [12],
 [5],
 [],
 [],
 [],
 [],
 [],
 [],
 [9],
 [5],
 [],
 [],
 [],
 [],
 [],
 [],
 [5, 12],
 [5, 12],
 [5],
 [9],
 [9],
 [9],
 [9],
 [24],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [],
 [],
 [],
 [],
 [17],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [],
 [29],
 [24, 29],
 [24],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [10],
 [10, 24],
 [],
 [20],
 [],
 [],
 [],
 [],
 [1, 27],
 [1],
 [1, 24],
 [24],
 [24],
 [12, 24],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [21],
 [21],
 [21],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [5],
 [20],
 [20],
 [20],
 [20

In [19]:
test_5s_dataset.to_parquet("/home/gil/comp0173/BirdSet/data/DataS1/metadata.jsonl")

Creating parquet from Arrow format:   0%|          | 0/61 [00:00<?, ?ba/s]

1572086

In [20]:
test_5s_dataset = load_dataset("parquet", data_files="/home/gil/comp0173/BirdSet/data/DataS1/metadata.jsonl")["train"]

Generating train split: 0 examples [00:00, ? examples/s]

In [22]:
test_5s_dataset["ebird_code_multilabel"]

[[],
 [],
 [2],
 [],
 [],
 [],
 [],
 [],
 [12],
 [],
 [],
 [],
 [2],
 [2],
 [],
 [],
 [],
 [],
 [21],
 [],
 [],
 [20],
 [20],
 [20],
 [21],
 [21],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [],
 [],
 [],
 [20, 21],
 [20],
 [],
 [],
 [],
 [],
 [],
 [18],
 [18],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [7],
 [7],
 [7],
 [7],
 [7],
 [7],
 [20],
 [20],
 [20],
 [20],
 [],
 [12],
 [5],
 [],
 [],
 [],
 [],
 [],
 [],
 [9],
 [5],
 [],
 [],
 [],
 [],
 [],
 [],
 [5, 12],
 [5, 12],
 [5],
 [9],
 [9],
 [9],
 [9],
 [24],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [],
 [],
 [],
 [],
 [17],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [],
 [29],
 [24, 29],
 [24],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [10],
 [10, 24],
 [],
 [20],
 [],
 [],
 [],
 [],
 [1, 27],
 [1],
 [1, 24],
 [24],
 [24],
 [12, 24],
 [20],
 [20],
 [20],
 [20],
 [20],
 [20],
 [21],
 [21],
 [21],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [],
 [5],
 [20],
 [20],
 [20],
 [20

## Join Datasets

In [3]:
dataset_DT_train = load_dataset("parquet", data_files="/home/gil/comp0173/BirdSet/data/DataS1_DT_train/metadata-train.parquet")["train"]
test_5s_dataset = load_dataset("parquet", data_files="/home/gil/comp0173/BirdSet/data/DataS1/metadata.parquet")["train"]

dataset_DT_train = dataset_DT_train.cast_column("audio", Audio(sampling_rate=32000, mono=True))
test_5s_dataset = test_5s_dataset.cast_column("audio", Audio(sampling_rate=32000, mono=True))

joined_dataset = DatasetDict({"train": dataset_DT_train,
                              "test_5s": test_5s_dataset
                              })

In [5]:
dataset_DT_train.features

{'filepath': Value(dtype='string', id=None),
 'start_time': Value(dtype='null', id=None),
 'end_time': Value(dtype='null', id=None),
 'low_freq': Value(dtype='null', id=None),
 'high_freq': Value(dtype='null', id=None),
 'lat': Value(dtype='float64', id=None),
 'long': Value(dtype='float64', id=None),
 'length': Value(dtype='int64', id=None),
 'call_type': Value(dtype='string', id=None),
 'sex': Value(dtype='string', id=None),
 'licence': Value(dtype='string', id=None),
 'local_time': Value(dtype='string', id=None),
 'ebird_code': Value(dtype='int64', id=None),
 'ebird_code_multilabel': Sequence(feature=Value(dtype='int64', id=None), length=-1, id=None),
 'ebird_code_secondary': Sequence(feature=Value(dtype='string', id=None), length=-1, id=None),
 'quality': Value(dtype='string', id=None),
 'microphone': Value(dtype='string', id=None),
 'source': Value(dtype='string', id=None),
 'detected_events': Sequence(feature=Sequence(feature=Value(dtype='float64', id=None), length=-1, id=None), 

In [4]:
test_5s_dataset.features

{'audio': Audio(sampling_rate=32000, mono=True, decode=True, id=None),
 'filepath': Value(dtype='string', id=None),
 'start_time': Value(dtype='float64', id=None),
 'end_time': Value(dtype='float64', id=None),
 'low_freq': Value(dtype='int64', id=None),
 'high_freq': Value(dtype='int64', id=None),
 'ebird_code': ClassLabel(names=['gcrfin', 'whcspa', 'amepip', 'sposan', 'rocwre', 'brebla', 'daejun', 'foxspa', 'clanut', 'moublu', 'casfin', 'mallar3', 'herthr', 'amerob', 'yerwar', 'yelwar', 'dusfly', 'mouchi', 'orcwar', 'warvir', 'norfli'], id=None),
 'ebird_code_multilabel': Sequence(feature=ClassLabel(names=['snogoo', 'pecsan', 'amgplo', 'batgod', 'cangoo', 'brant', 'gwfgoo', 'pursan', 'speeid', 'lotduc', 'pacloo', 'snobun', 'lotjae', 'dunlin', 'whrsan', 'tunswa', 'arcter', 'rudtur', 'lobdow', 'retloo', 'laplon', 'semsan', 'bkbplo', 'baisan', 'kineid', 'pomjae', 'sabgul', 'redpha1', 'sander', 'comrav', 'semplo'], id=None), length=-1, id=None),
 'ebird_code_secondary': Sequence(feature=V